# Практика 2. Эмбеддинги вершин своими руками

План на сегодня:
0. знакомимся с данными: что за граф Cora и из чего он состоит;
1. случайные блуждания, включая node2vec с параметрами `p` и `q`;
2. skip-gram с negative sampling — обучаем без gensim, на голом PyTorch;
3. смотрим на эмбеддинги глазами и проверяем их на двух задачах;
4. предсказание связей с честным сплитом и сравнение с эвристикой Adamic–Adar.

Никаких графовых нейросетей сегодня не будет — они начнутся на следующей практике.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

try:
    import torch_geometric  # noqa: F401
except ImportError:
    !pip install -q torch_geometric

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

rng = np.random.default_rng(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx

dataset = Planetoid(root="/tmp/Cora", name="Cora")
data = dataset[0]
G = to_networkx(data, to_undirected=True)
N = data.num_nodes
y = data.y.numpy()
neighbors = [sorted(G.neighbors(v)) for v in range(N)]
print(f"Cora: {N} вершин, {G.number_of_edges()} рёбер")

## 0. Знакомимся с Cora

Cora — граф цитирований научных статей по машинному обучению:

- **вершина** — статья;
- **ребро** — одна статья ссылается на другую; направление ссылки мы отбрасываем, граф неориентированный;
- **признаки `x`** — по 1433 нуля и единицы на статью: есть ли в её тексте слово из словаря;
- **метка `y`** — тематика статьи, одна из семи: нейросети, генетические алгоритмы, вероятностные
  методы, обучение с подкреплением, обучение правил, теория и рассуждения по прецедентам.
  В PyG тематики закодированы числами от 0 до 6.

Сегодня модель признаков `x` не видит: shallow-эмбеддинги учатся только по структуре графа.
Метки `y` понадобятся в самом конце — проверить, выучили ли эмбеддинги что-то осмысленное.

In [ ]:
print(data)
print()
print(f"x:          {tuple(data.x.shape)} — строка на статью, столбец на слово из словаря")
print(f"edge_index: {tuple(data.edge_index.shape)} — пары (откуда, куда); "
      f"{data.edge_index.shape[1]} записей = {G.number_of_edges()} рёбер × 2, "
      f"каждое ребро записано в обе стороны")
print(f"y:          {tuple(data.y.shape)} — тематика статьи, числа от 0 до {int(data.y.max())}")
print("train_mask, val_mask, test_mask — стандартный сплит для классификации, сегодня не нужен")

In [ ]:
# одна статья под лупой
paper = 0
words = data.x[paper].nonzero().flatten().tolist()
print(f"статья {paper}: тематика {y[paper]}, слов из словаря: {len(words)} из {data.num_features}")
print(f"номера этих слов: {words}")
print(f"ссылки (соседи в графе): {neighbors[paper]}")
print(f"тематики соседей:        {[int(y[u]) for u in neighbors[paper]]}")

Сами слова в датасете не сохранились — только их номера в словаре. Поэтому признаки статьи
для нас просто набор включённых «лампочек».

In [ ]:
degrees = np.array([len(nbrs) for nbrs in neighbors])
per_class = np.bincount(y)
same_topic = np.mean([y[a] == y[b] for a, b in G.edges()])
print(f"ссылок у статьи: медиана {np.median(degrees):.0f}, среднее {degrees.mean():.1f}, "
      f"максимум {degrees.max()}")
print(f"статей по тематикам 0–6: {per_class.tolist()}")
print(f"рёбер между статьями одной тематики: {same_topic:.0%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.hist(degrees, bins=range(1, degrees.max() + 2), color="#1F3A5F")
ax1.set_yscale("log"); ax1.set_xlabel("число ссылок у статьи"); ax1.set_ylabel("статей")
ax1.set_title("У большинства статей всего несколько ссылок")
ax2.bar(range(len(per_class)), per_class, color="#0E8F8F")
ax2.set_xlabel("тематика"); ax2.set_ylabel("статей"); ax2.set_title("Тематики по размеру")
plt.tight_layout(); plt.show()

Что из этого важно для сегодняшней практики:

- у типичной статьи всего несколько ссылок, так что из одной вершины блуждание может
  пойти лишь в пару мест — поэтому блужданий из каждой вершины делаем много;
- большинство рёбер соединяет статьи одной тематики: статьи ссылаются на свою область.
  Это и есть гомофилия из лекции 1 — и причина, по которой эмбеддинги, выученные по одной
  структуре, потом угадывают тематику.

## 1. Случайные блуждания

Начнём с самого простого варианта: на каждом шаге идём в случайного соседа.

In [ ]:
# TODO: реализуйте равномерное случайное блуждание
# start — стартовая вершина, length — сколько вершин должно быть в блуждании
def random_walk(start, length):
    walk = [start]
    ...
    return walk

print(random_walk(0, 10))

Теперь node2vec. Отличие в том, что вероятность следующего шага зависит от того,
откуда мы пришли:

- вернуться назад — вес `1/p`;
- пойти к соседу предыдущей вершины (остаться «рядом») — вес `1`;
- уйти дальше — вес `1/q`.

Большое `q` держит блуждание около старта, маленькое — уводит вглубь.

In [ ]:
# TODO: реализуйте блуждание node2vec
def node2vec_walk(start, length, p=1.0, q=1.0):
    walk = [start]
    while len(walk) < length:
        current = walk[-1]
        nbrs = neighbors[current]
        if not nbrs:
            break
        if len(walk) == 1:
            walk.append(int(rng.choice(nbrs)))
            continue
        prev = walk[-2]
        weights = ...
        probs = ...
        walk.append(int(rng.choice(nbrs, p=probs)))
    return walk

print("p=1, q=1   ", node2vec_walk(0, 10))
print("p=1, q=4   ", node2vec_walk(0, 10, q=4))
print("p=1, q=0.25", node2vec_walk(0, 10, q=0.25))

## 2. Из блужданий — в обучающие пары

Каждое блуждание режем окном: для центральной вершины все вершины в окне считаются
её контекстом.

In [ ]:
# TODO: соберите пары (центр, контекст) из блужданий
# walks_per_node блужданий длины walk_len из каждой вершины, окно window
def build_pairs(walks_per_node=5, walk_len=10, window=3, p=1.0, q=1.0):
    pairs = []
    for _ in range(walks_per_node):
        for v in range(N):
            walk = node2vec_walk(v, walk_len, p, q)
            ...
    return np.array(pairs, dtype=np.int64)

pairs = build_pairs()
print("пар для обучения:", len(pairs))

## 3. Skip-gram с negative sampling

Обучаем две таблицы векторов: `emb_in` для центральной вершины и `emb_out` для контекста.
Лосс из лекции:

$$\mathcal{L} = -\log \sigma(z_u^\top z'_v) - \sum_k \log \sigma(-z_u^\top z'_{n_k})$$

Негативы берём пропорционально степени в степени 0.75.

Весь технический каркас — оптимизатор, батчи, сэмплирование негативов — уже написан ниже.
Ваша задача одна: перевести формулу лосса в три строки кода.

Подсказки:
- `torch.nn.functional.logsigmoid(x)` — это `log(sigmoid(x))`;
- `(a * b).sum(-1)` — скалярное произведение по последней оси;
- у позитивной пары форма `[B, dim]`, у негативов `[B, neg, dim]`,
  поэтому центральный вектор нужно «растянуть»: `z_c.unsqueeze(1)`.

In [ ]:
from torch.nn.functional import logsigmoid

def train_embeddings(pairs, dim=64, epochs=3, neg=5, batch_size=2048, lr=0.01):
    emb_in = torch.nn.Embedding(N, dim).to(device)
    emb_out = torch.nn.Embedding(N, dim).to(device)
    opt = torch.optim.Adam(list(emb_in.parameters()) + list(emb_out.parameters()), lr=lr)

    # негативы сэмплируем пропорционально степени в степени 0.75 — это уже готово
    deg = np.array([len(neighbors[v]) for v in range(N)], dtype=float)
    noise = torch.tensor(deg ** 0.75 / (deg ** 0.75).sum(), dtype=torch.float, device=device)

    pairs_t = torch.tensor(pairs, device=device)
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(len(pairs_t), device=device)
        total = 0.0
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            center, context = pairs_t[idx, 0], pairs_t[idx, 1]
            negatives = torch.multinomial(noise, len(idx) * neg, replacement=True).view(len(idx), neg)

            z_c = emb_in(center)                    # [B, dim]  вектор центральной вершины
            z_ctx = emb_out(context)                # [B, dim]  вектор соседа по блужданию
            z_neg = emb_out(negatives)              # [B, neg, dim]  случайные вершины

            # TODO: посчитайте лосс из лекции
            #   L = -log sigmoid(z_c . z_ctx) - sum_k log sigmoid(-z_c . z_neg_k)
            pos_score = ...          # [B]
            neg_score = ...          # [B]  сумма по всем негативам
            loss = ...               # среднее по батчу

            opt.zero_grad(); loss.backward(); opt.step()
            total += float(loss) * len(idx)
        history.append(total / len(pairs_t))
        print(f"эпоха {epoch + 1}: loss {history[-1]:.4f}")
    return emb_in.weight.detach().cpu().numpy(), emb_out.weight.detach().cpu().numpy(), history

Z_in, Z_out, history = train_embeddings(pairs)
print("эмбеддинги:", Z_in.shape)

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, len(history) + 1), history, "o-", color="#1F3A5F", linewidth=2)
plt.xlabel("эпоха"); plt.ylabel("loss"); plt.title("Обучение эмбеддингов"); plt.show()

## 4. Смотрим на результат

In [ ]:
proj = PCA(n_components=2).fit_transform(Z_in)
plt.figure(figsize=(7, 5))
plt.scatter(proj[:, 0], proj[:, 1], c=y, cmap="tab10", s=8)
plt.title("эмбеддинги Cora, PCA в 2D; цвет — тематика статьи")
plt.xticks([]); plt.yticks([]); plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def evaluate(X, name):
    scores = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=cv)
    print(f"{name}: accuracy {scores.mean():.3f} ± {scores.std():.3f}")
    return scores.mean()

evaluate(Z_in, "эмбеддинги из блужданий")
evaluate(data.x.numpy(), "признаки статьи (слова)")
evaluate(np.hstack([Z_in, data.x.numpy()]), "эмбеддинги + слова")

Обратите внимание: эмбеддинги обучались **без единой метки** — только по структуре
цитирований. Тематика статьи в них проявилась сама.

## 5. Предсказание связей

Теперь главное упражнение практики. Спрячем часть рёбер и попробуем их восстановить.

Сплит рёбер — место, где чаще всего появляется утечка. Функция ниже делает всё правильно:
прячет часть рёбер, набирает столько же несуществующих пар и возвращает граф **без**
спрятанных рёбер. Прочитайте её внимательно — в домашнем задании такую придётся писать самим.

In [ ]:
def split_edges(graph, test_ratio=0.1, seed=0):
    """Прячет часть рёбер: возвращает обучающий граф, тестовые рёбра и негативы."""
    r = np.random.default_rng(seed)
    all_edges = np.array([(u, v) for u, v in graph.edges()])
    n_test = int(test_ratio * len(all_edges))

    chosen = r.choice(len(all_edges), size=n_test, replace=False)
    positives = all_edges[chosen]

    train_graph = graph.copy()
    train_graph.remove_edges_from([tuple(e) for e in positives])

    existing = {(u, v) for u, v in all_edges} | {(v, u) for u, v in all_edges}
    negatives = []
    while len(negatives) < n_test:
        u, v = r.integers(0, graph.number_of_nodes(), size=2)
        if u != v and (u, v) not in existing:
            negatives.append((int(u), int(v)))
    return train_graph, positives, np.array(negatives)

train_graph, test_pos, test_neg = split_edges(G)
print(f"спрятали рёбер: {len(test_pos)}, негативов столько же")
print(f"в обучающем графе осталось: {train_graph.number_of_edges()} рёбер из {G.number_of_edges()}")

In [ ]:
# переобучаем эмбеддинги ТОЛЬКО на обучающем графе — иначе получим утечку
neighbors = [sorted(train_graph.neighbors(v)) for v in range(N)]
pairs_lp = build_pairs()
Z_in_lp, Z_out_lp, _ = train_embeddings(pairs_lp)

y_true = np.r_[np.ones(len(test_pos)), np.zeros(len(test_neg))]
test_all = np.vstack([test_pos, test_neg])

Скор пары считаем ровно так же, как при обучении: вектор вершины `u` из таблицы центров
(`Z_in_lp`) на вектор вершины `v` из таблицы соседей (`Z_out_lp`). Сравним три способа:

1. **скалярное произведение** этих двух векторов — то, что оптимизировал лосс;
2. **косинус** между теми же векторами — то же сравнение, но без длин;
3. **Adamic–Adar** по обучающему графу — эвристика из лекции 1, эмбеддинги ей не нужны.

In [ ]:
# TODO: посчитайте ROC-AUC для трёх способов сравнить вершины
def auc(scores, name):
    value = roc_auc_score(y_true, scores)
    results[name] = value
    print(f"{name}: ROC-AUC {value:.3f}")
    return value

results = {}
u, v = test_all[:, 0], test_all[:, 1]
zu, zv = Z_in_lp[u], Z_out_lp[v]
# 1. скалярное произведение zu и zv
# 2. косинус между zu и zv
# 3. Adamic-Adar: nx.adamic_adar_index(train_graph, пары) возвращает тройки (a, b, скор)
...

In [ ]:
plt.figure(figsize=(7, 3.2))
names = list(results)
values = [results[n] for n in names]
colors = ["#E07A3F" if v == max(values) else "#1F3A5F" for v in values]
bars = plt.bar(names, values, color=colors)
for rect, v in zip(bars, values):
    plt.text(rect.get_x() + rect.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
plt.axhline(0.5, color="#6B7280", linestyle=":", label="случайное угадывание")
plt.ylabel("ROC-AUC"); plt.ylim(0, 1.05); plt.legend(frameon=False)
plt.xticks(rotation=15, ha="right"); plt.title("Способы сравнить две вершины")
plt.tight_layout(); plt.show()

In [ ]:
# почему Adamic-Adar отстаёт: сколько спрятанных рёбер вообще без общих соседей
common = [len(list(nx.common_neighbors(train_graph, int(a), int(b)))) for a, b in test_pos]
print(f"спрятанных рёбер без единого общего соседа: {np.mean(np.array(common) == 0):.0%}")

## 6. Выводы и вопросы

Сравните числа. Скорее всего, вы увидите примерно такую картину (точные значения
зависят от сида и железа):

- скалярное произведение и косинус почти равны — длины векторов добавляют совсем немного;
- оба заметно лучше Adamic–Adar. Посмотрите на последнюю ячейку: у большой доли спрятанных
  рёбер нет ни одного общего соседа, и для эвристики такие пары неотличимы от случайных.
  Эмбеддинги видят дальше: в пары для обучения попадают вершины в нескольких шагах по блужданию.

**Главный урок:** эвристика — обязательный baseline, даже когда кажется, что эмбеддинги
очевидно сильнее. На Cora они её обгоняют, но на плотном графе, где у соседей много общих
знакомых, бывает наоборот — вспомните Les Misérables из лекции.

### Задания для самостоятельной работы
1. Обучите эмбеддинги с `q = 4` и `q = 0.25`. Какая настройка лучше для классификации
   вершин, а какая для предсказания связей? Почему?
2. Постройте зависимость качества от размерности: 8, 32, 128, 256.
3. Замените негативы на равномерные (без степени 0.75) и сравните.
4. Попробуйте обучить логистическую регрессию поверх поэлементного произведения
   эмбеддингов вместо скалярного произведения — станет ли лучше?